### Imports

In [1]:
import os
import json
import math
import random
import pickle
import time
from collections import defaultdict, Counter
from tqdm import tqdm
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

CONFIG = {
    "tokenized_json": "telugu_tokenized_sentences.json",
    "vocab_txt": "telugu_vocabulary.txt",
    "ngram_n": 3,
    "train_samples": 20000,
    "top_k": 12,
    "negs_per_pos": 2,
    "vocab_size": 30000,
    "embed_dim": 256,
    "hidden_size": 256,
    "batch_size": 64,
    "epochs": 5,
    "lr": 1e-3,
    "seed": 42,
    "device": "cpu",
}
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

### Load tokenized corpus and vocabulary

In [2]:
def load_tokenized_corpus(path):

    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found.")
    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not data:
        raise ValueError("Empty corpus.")
    
    if isinstance(data[0], list):
        return data
    
    return [line.split() for line in data]

def load_vocab(path):

    if path and os.path.exists(path):

        with open(path, "r", encoding="utf-8") as f:
            words = [line.strip() for line in f if line.strip()]

        return words
    
    return None

corpus = load_tokenized_corpus(CONFIG["tokenized_json"])
print("Loaded corpus size:", len(corpus))

vocab_words = load_vocab(CONFIG.get("vocab_txt"))

if vocab_words:
    print("Loaded vocabulary size from file:", len(vocab_words))
else:
    print("No vocab file provided; will build from corpus (top K).")

Loaded corpus size: 1036856
Loaded vocabulary size from file: 959816


### Build / limit vocabulary (top-K) and mappings

In [3]:
def build_vocab_from_corpus(corpus, top_k):
    ctr = Counter()
    for toks in corpus:
        ctr.update(toks)
    most = [w for w, _ in ctr.most_common(top_k)]
    return most, ctr

if vocab_words is None:
    vocab_words, unigram_counts = build_vocab_from_corpus(corpus, CONFIG["vocab_size"])
else:
    unigram_counts = Counter()
    for toks in corpus:
        unigram_counts.update(toks)
    vocab_words = vocab_words[:CONFIG["vocab_size"]]

PAD = "<pad>"
UNK = "<unk>"
VOCAB = [PAD, UNK] + vocab_words
word2idx = {w: i for i, w in enumerate(VOCAB)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(VOCAB)
print("Final vocab size (with PAD,UNK):", vocab_size)

Final vocab size (with PAD,UNK): 30002


### Scratch N-gram model (train & predict_next)

In [8]:
class NGramModel:
    def __init__(self, n=3):
        self.n = n
        self.ngrams = defaultdict(Counter)
        self.unigram_counts = Counter()
        self.vocab = set()

    def train(self, tokenized_sentences):
        pad = ["<s>"] * (self.n - 1)
        for toks in tqdm(tokenized_sentences, desc="Training N-gram"):
            for w in toks:
                self.unigram_counts[w] += 1
                self.vocab.add(w)
            seq = pad + toks + ["</s>"]
            for i in range(len(seq) - self.n + 1):
                ctx = tuple(seq[i:i + self.n - 1])
                tgt = seq[i + self.n - 1]
                self.ngrams[ctx][tgt] += 1

    def predict_next(self, context_tokens, top_k=5, lambda_ngram=0.9, lambda_unigram=0.1):
        scores = {}
        if len(context_tokens) >= self.n - 1:
            ctx = tuple(context_tokens[-(self.n - 1):])
            data = self.ngrams.get(ctx, {})
            total = sum(data.values())
            if total > 0:
                for w,c in data.items():
                    if w in ("<s>", "</s>"): continue
                    scores[w] = (c/total) * lambda_ngram
        unigram_total = sum(self.unigram_counts.values()) or 1
        for w,c in self.unigram_counts.most_common(200):
            if w in scores: continue
            scores[w] = (c/unigram_total) * lambda_unigram
        items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [w for w,_ in items[:top_k]]

ngram = NGramModel(n=CONFIG["ngram_n"])
ngram.train(corpus)
print("N-gram vocab size:", len(ngram.vocab))
print("N-gram perplexity (quick estimate):", round(ngram.perplexity(corpus), 3) if hasattr(ngram, "perplexity") else "N/A")


Training N-gram: 100%|██████████| 1036856/1036856 [00:26<00:00, 38923.56it/s]

N-gram vocab size: 959816
N-gram perplexity (quick estimate): N/A


### Build training pairs (context, candidate, label)

In [ ]:
def sample_training_pairs(corpus, ngram_model, train_samples, top_k, negs_per_pos, seed=42):
    random.seed(seed)
    eligible_idx = [i for i, t in enumerate(corpus) if len(t) >= 2]
    sampled_idx = eligible_idx if train_samples >= len(eligible_idx) else random.sample(eligible_idx, k=min(train_samples, len(eligible_idx)))
    pairs = []
    
    for i in tqdm(sampled_idx, desc="sampling contexts"):
        toks = corpus[i]
        ctx_len = min(4, len(toks) - 1)
        ctx = toks[:ctx_len]
        gold = toks[ctx_len]
        cands = ngram_model.predict_next(ctx, top_k=top_k)
        
        if gold not in cands:
            if len(cands) < top_k:
                cands.append(gold)
            else:
                cands[-1] = gold
        
        for c in cands:
            lbl = 1 if c == gold else 0
            p_ng = 1e-12
            ctx_tuple = tuple(ctx[-(ngram_model.n - 1):]) if len(ctx) >= ngram_model.n - 1 else tuple(ctx)
            data = ngram_model.ngrams.get(ctx_tuple, {})
            total = sum(data.values())
            if total > 0:
                p_ng = data.get(c, 0) / total
            meta = (math.log(p_ng + 1e-12), math.log(unigram_counts.get(c, 0) + 1))
            pairs.append((ctx, c, lbl, meta))
            
        if negs_per_pos > 0:
            candidates_pool = [w for w, _ in unigram_counts.most_common(1000) if w not in cands]
            for neg in candidates_pool[:negs_per_pos]:
                p_ng = 1e-12
                meta = (math.log(p_ng), math.log(unigram_counts.get(neg, 0) + 1))
                pairs.append((ctx, neg, 0, meta))
                
    return pairs

pairs = sample_training_pairs(corpus, ngram, CONFIG["train_samples"], CONFIG["top_k"], CONFIG["negs_per_pos"], seed=CONFIG["seed"])
print("Total training pairs:", len(pairs))

sampling contexts: 100%|██████████| 20000/20000 [51:21<00:00,  6.49it/s] 

Total training pairs: 280000


### Dataset & DataLoader (maps tokens -> indices; pads contexts)

In [ ]:
class RerankerDataset(Dataset):
    def __init__(self, pairs, word2idx, max_ctx_len=4):
        self.pairs = pairs
        self.w2i = word2idx
        self.max_ctx_len = max_ctx_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ctx, cand, lbl, meta = self.pairs[idx]
        ctx_idx = [self.w2i.get(w, self.w2i[UNK]) for w in ctx[-self.max_ctx_len:]]
        pad_len = self.max_ctx_len - len(ctx_idx)
        ctx_idx = [self.w2i[PAD]] * pad_len + ctx_idx
        cand_idx = self.w2i.get(cand, self.w2i[UNK])
        meta_feats = np.array(meta, dtype=np.float32)
        return np.array(ctx_idx, dtype=np.int64), np.int64(cand_idx), np.int64(lbl), meta_feats

def collate_batch(batch):
    ctxs = np.stack([b[0] for b in batch])
    cands = np.array([b[1] for b in batch], dtype=np.int64)
    labels = np.array([b[2] for b in batch], dtype=np.float32)
    metas = np.stack([b[3] for b in batch])
    return torch.from_numpy(ctxs), torch.from_numpy(cands), torch.from_numpy(labels), torch.from_numpy(metas)

dataset = RerankerDataset(pairs, word2idx, max_ctx_len=4)
loader = DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_batch, drop_last=False)

### GRU Reranker model (PyTorch)

In [4]:
class GRUReranker(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden, num_layers=1, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(hidden + emb_dim + 2, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, ctx_idx, cand_idx, meta_feats):
        emb_ctx = self.emb(ctx_idx)
        _, h = self.gru(emb_ctx)
        h = h.squeeze(0)
        emb_c = self.emb(cand_idx)
        x = torch.cat([h, emb_c, meta_feats], dim=1)
        logits = self.mlp(x).squeeze(1)
        return logits

### Training loop (train + quick val)

In [ ]:
from torch.utils.data import DataLoader

device = torch.device(CONFIG["device"])
model = GRUReranker(vocab_size, emb_dim=CONFIG["embed_dim"], hidden=CONFIG["hidden_size"]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
loss_fn = nn.BCEWithLogitsLoss()

val_portion = int(0.05 * len(pairs))
train_pairs = pairs[val_portion:]
val_pairs = pairs[:val_portion] if val_portion>0 else []

train_ds = RerankerDataset(train_pairs, word2idx)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_batch)


def evaluate_on_pairs_using_dataloader(model, val_pairs, word2idx, batch_size=CONFIG["batch_size"]):
    if not val_pairs:
        return 0.0
    model.eval()
    val_ds = RerankerDataset(val_pairs, word2idx, max_ctx_len=4)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
    correct = 0
    total = 0
    with torch.no_grad():
        for ctxs, cands, labels, metas in val_loader:
            ctxs = ctxs.to(device); cands = cands.to(device); labels = labels.to(device); metas = metas.to(device)
            logits = model(ctxs, cands, metas)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()
            correct += (preds.cpu().numpy() == labels.cpu().numpy()).sum()
            total += labels.size(0)
    return correct / total if total else 0.0


print("Training reranker ...")
for epoch in range(CONFIG["epochs"]):
    model.train()
    t0 = time.time()
    epoch_loss = 0.0
    for ctxs, cands, labels, metas in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}"):
        ctxs, cands, labels, metas = ctxs.to(device), cands.to(device), labels.to(device), metas.to(device)
        opt.zero_grad()
        logits = model(ctxs, cands, metas)
        loss = loss_fn(logits, labels)
        loss.backward()
        opt.step()
        epoch_loss += float(loss.item()) * ctxs.size(0)
    epoch_loss /= len(train_loader.dataset)
    val_acc = evaluate_on_pairs_using_dataloader(model, val_pairs, word2idx) if val_pairs else 0.0
    print(f"Epoch {epoch+1}/{CONFIG['epochs']}  loss={epoch_loss:.4f}  val_acc={val_acc:.4f}  time={time.time()-t0:.1f}s")

torch.save({"model_state": model.state_dict(), "word2idx": word2idx, "idx2word": idx2word}, "gru_reranker.pt")
print("Saved GRU reranker to gru_reranker.pt")


Training reranker ...


Epoch 1/5: 100%|██████████| 4157/4157 [04:52<00:00, 14.20it/s]


Epoch 1/5  loss=0.0810  val_acc=0.9748  time=295.0s


Epoch 2/5: 100%|██████████| 4157/4157 [07:26<00:00,  9.32it/s]


Epoch 2/5  loss=0.0660  val_acc=0.9759  time=448.8s


...

Epoch 3/5  loss=0.0639  val_acc=0.9754  time=273.6s


Epoch 4/5: 100%|██████████| 4157/4157 [06:07<00:00, 11.30it/s]


Epoch 4/5  loss=0.0629  val_acc=0.9754  time=370.1s


Epoch 5/5: 100%|██████████| 4157/4157 [05:19<00:00, 12.99it/s]


Epoch 5/5  loss=0.0622  val_acc=0.9749  time=320.8s
Saved GRU reranker to gru_reranker.pt


### Inference helpers & safe step-by-step completion (uses GRU scores)

In [5]:
def score_candidates_for_context(ctx_tokens, candidates, model, word2idx, device):

    max_ctx_len = 4
    
    ctx_idx = [word2idx.get(w, word2idx[UNK]) for w in ctx_tokens[-max_ctx_len:]]
    pad_len = max_ctx_len - len(ctx_idx)
    ctx_idx = [word2idx[PAD]] * pad_len + ctx_idx
    ctx_batch = torch.tensor([ctx_idx] * len(candidates), dtype=torch.int64).to(device)
    cand_idx = torch.tensor([word2idx.get(c, word2idx[UNK]) for c in candidates], dtype=torch.int64).to(device)
    
    metas = []
    
    for c in candidates:
        p_ng = 1e-12
        ctx_tuple = tuple(ctx_tokens[-(ngram.n - 1):]) if len(ctx_tokens) >= ngram.n - 1 else tuple(ctx_tokens)
        data = ngram.ngrams.get(ctx_tuple, {})
        tot = sum(data.values())
        if tot > 0:
            p_ng = data.get(c, 0) / tot
        metas.append([math.log(p_ng + 1e-12), math.log(unigram_counts.get(c, 0) + 1)])
    
    metas = torch.tensor(metas, dtype=torch.float32).to(device)
    model.eval()
    
    with torch.no_grad():
        logits = model(ctx_batch, cand_idx, metas)
        probs = torch.sigmoid(logits).cpu().numpy().tolist()
    return list(zip(candidates, probs))


def complete_sentence_gru_fluent(prefix_text, ngram, model, word2idx, idx2word, device,
                                 max_steps=20, top_k=CONFIG["top_k"],
                                 ban_window=4, min_score=0.03, alpha=0.15,
                                 cumulative_penalty=0.85, sample_T=None, show_top=4):
    prefix_tokens = prefix_text if isinstance(prefix_text, list) else (prefix_text.split() if isinstance(prefix_text, str) else [])
    if not prefix_tokens:
        print("No tokens in prefix.")
        return []

    print("Prefix tokens:", prefix_tokens)
    
    completed = list(prefix_tokens)
    step = 0

    def softmax_with_temp(x, T=1.0):
        x = np.array(x, dtype=float)
        x = x - np.max(x)
        ex = np.exp(x / max(T, 1e-8))
        denom = ex.sum()
        if denom <= 0:
            return np.ones_like(ex) / len(ex)
        return ex / denom

    while step < max_steps:
        step += 1
        candidates = ngram.predict_next(completed, top_k=top_k)
        if not candidates:
            print("\nNo candidates from n-gram. Stopping.")
            break

        scored = score_candidates_for_context(completed, candidates, model, word2idx, device)
        scored = [(w, p * ((unigram_counts.get(w, 0) + 1) ** alpha)) for (w, p) in scored]
        scored = [(w, p * (cumulative_penalty ** completed.count(w))) for (w, p) in scored]

        banned = {completed[-1]} if len(completed) > 0 else set()
        filtered = [(w, p) for (w, p) in scored if w not in banned]
        if not filtered:
            filtered = scored.copy()

        filtered = sorted(filtered, key=lambda x: x[1], reverse=True)
        ctx_display = " ".join(completed[-(ngram.n - 1):]) if len(completed) >= (ngram.n - 1) else " ".join(completed)
        print(f"\nStep {step} — context: {ctx_display}")

        cands, raw_scores = zip(*filtered)
        raw_scores = np.array(raw_scores, dtype=float)
        raw_scores = np.clip(raw_scores, 1e-12, None)

        if sample_T and sample_T > 0:
            probs = softmax_with_temp(raw_scores, sample_T)
        else:
            s = raw_scores.sum()
            probs = (raw_scores / s) if s > 0 else np.ones_like(raw_scores) / len(raw_scores)

        for w, p in zip(cands[:show_top], probs[:show_top]):
            print(f"   {w.ljust(18)} -> {p:.6f}")

        if sample_T and sample_T > 0:
            pick = np.random.choice(list(cands), p=probs)
        else:
            pick = cands[int(np.argmax(probs))]

        pick = str(pick)
        print("Selected sample:", pick)

        top_prob = float(probs[0]) if len(probs) > 0 else 0.0
        if top_prob < min_score:
            print("Top normalized probability below min_score — stopping.")
            break

        if pick in ("</s>", ".", "?", "।"):
            break

        if len(completed) >= max_steps + len(prefix_tokens):
            print("Reached max length. Stopping.")
            break

        completed.append(pick)

    print("\nCompleted sentence:", " ".join(completed))
    return completed

### Quick usage / demo

In [6]:
checkpoint = torch.load("gru_reranker.pt", map_location=CONFIG["device"])
model = GRUReranker(vocab_size, emb_dim=CONFIG["embed_dim"], hidden=CONFIG["hidden_size"])
model.load_state_dict(checkpoint["model_state"])
model.to(CONFIG["device"])
word2idx = checkpoint["word2idx"]
idx2word = checkpoint["idx2word"]

In [14]:
prefix = "సైనికులు"
complete_sentence_gru_fluent(prefix, ngram, model, word2idx, idx2word, CONFIG["device"],sample_T= None, max_steps=10)

# Many soldiers have their own

Prefix tokens: ['సైనికులు']

Step 1 — context: సైనికులు
   చాలా               -> 0.145521
   నీ                 -> 0.135873
   లో                 -> 0.091067
   తన                 -> 0.080541
Selected sample: చాలా

Step 2 — context: సైనికులు చాలా
   మంది               -> 0.999482
   కూడా               -> 0.000052
   ఆమె                -> 0.000052
   అన్నాడు            -> 0.000052
Selected sample: మంది

Step 3 — context: చాలా మంది
   తమ                 -> 0.099291
   ఉన్నారు            -> 0.099064
   పెద్ద              -> 0.098406
   వున్నారు           -> 0.091047
Selected sample: తమ

Step 4 — context: మంది తమ
   దగ్గర              -> 0.119554
   జాతీయ              -> 0.100186
   పిల్ల              -> 0.095984
   ప్రాణాలు           -> 0.093761
Selected sample: దగ్గర

Step 5 — context: తమ దగ్గర
   ఉన్న               -> 0.389728
   వున్నది            -> 0.069301
   సెల్               -> 0.065973
   చదివిన             -> 0.064924
Selected sample: ఉన్న

Step 6 — context: దగ్గర ఉన్న
   అందుక

['సైనికులు',
 'చాలా',
 'మంది',
 'తమ',
 'దగ్గర',
 'ఉన్న',
 'అందుకోమని',
 'మరొక',
 'పని',
 'చాలా',
 'వుంది']

In [10]:
prefix = "చంపబడిన వారిలో"
complete_sentence_gru_fluent(prefix, ngram, model, word2idx, idx2word, CONFIG["device"],sample_T= None, max_steps=5)

# Among those killed was Saint John, the commander of the fortress.

Prefix tokens: ['చంపబడిన', 'వారిలో']

Step 1 — context: చంపబడిన వారిలో
   కోట                -> 0.999241
   కూడా               -> 0.000077
   ఆమె                -> 0.000076
   అన్నాడు            -> 0.000076
Selected sample: కోట

Step 2 — context: వారిలో కోట
   కమాండర్            -> 0.999651
   కూడా               -> 0.000035
   ఆమె                -> 0.000034
   అన్నాడు            -> 0.000034
Selected sample: కమాండర్

Step 3 — context: కోట కమాండర్
   అయిన               -> 0.602716
   కల్నల్             -> 0.396992
   కూడా               -> 0.000031
   ఆమె                -> 0.000030
Selected sample: అయిన

Step 4 — context: కమాండర్ అయిన
   సెయింట్            -> 0.999709
   కూడా               -> 0.000029
   ఆమె                -> 0.000029
   అన్నాడు            -> 0.000028
Selected sample: సెయింట్

Step 5 — context: అయిన సెయింట్
   జాన్               -> 0.999753
   కూడా               -> 0.000024
   ఆమె                -> 0.000024
   అన్నాడు            -> 0.000024
Selected sample: జాన్

Complete

['చంపబడిన', 'వారిలో', 'కోట', 'కమాండర్', 'అయిన', 'సెయింట్', 'జాన్']

In [11]:
prefix = "శతాబ్దాలు గడిచేటప్పటికి"
complete_sentence_gru_fluent(prefix, ngram, model, word2idx, idx2word, CONFIG["device"],sample_T= 0.3, max_steps=5)

# Over the centuries, the various processes in papermaking have evolved greatly.

Prefix tokens: ['శతాబ్దాలు', 'గడిచేటప్పటికి']

Step 1 — context: శతాబ్దాలు గడిచేటప్పటికి
   కాగితం             -> 0.997859
   కూడా               -> 0.000195
   ఆమె                -> 0.000195
   అన్నాడు            -> 0.000195
Selected sample: కాగితం

Step 2 — context: గడిచేటప్పటికి కాగితం
   నిర్మాణంలో         -> 0.991739
   కూడా               -> 0.000751
   ఆమె                -> 0.000751
   అన్నాడు            -> 0.000751
Selected sample: నిర్మాణంలో

Step 3 — context: కాగితం నిర్మాణంలో
   వివిధ              -> 0.999608
   కూడా               -> 0.000036
   ఆమె                -> 0.000036
   అన్నాడు            -> 0.000036
Selected sample: వివిధ

Step 4 — context: నిర్మాణంలో వివిధ
   ప్రక్రియలు         -> 0.974747
   కూడా               -> 0.002296
   ఆమె                -> 0.002296
   అన్నాడు            -> 0.002296
Selected sample: ప్రక్రియలు

Step 5 — context: వివిధ ప్రక్రియలు
   చాలా               -> 0.999993
   కూడా               -> 0.000001
   ఆమె                -> 0.000001
   అన్నాడు   

['శతాబ్దాలు',
 'గడిచేటప్పటికి',
 'కాగితం',
 'నిర్మాణంలో',
 'వివిధ',
 'ప్రక్రియలు',
 'చాలా']

In [12]:
prefix = "ఆయన"
complete_sentence_gru_fluent(prefix, ngram, model, word2idx, idx2word, CONFIG["device"],sample_T= None, max_steps=5)

# He has a good name and reputation among his peers

Prefix tokens: ['ఆయన']

Step 1 — context: ఆయన
   చాలా               -> 0.145521
   నీ                 -> 0.135873
   లో                 -> 0.091067
   తన                 -> 0.080541
Selected sample: చాలా

Step 2 — context: ఆయన చాలా
   మంచి               -> 0.114485
   బాగా               -> 0.111603
   గొప్ప              -> 0.110035
   మామూలుగా           -> 0.093220
Selected sample: మంచి

Step 3 — context: చాలా మంచి
   పేరు               -> 0.100194
   మనిషి              -> 0.096163
   అమ్మాయి            -> 0.094697
   పని                -> 0.094530
Selected sample: పేరు

Step 4 — context: మంచి పేరు
   కూడా               -> 0.129930
   వచ్చింది           -> 0.118160
   రావడం              -> 0.091363
   తెచ్చిన            -> 0.083877
Selected sample: కూడా

Step 5 — context: పేరు కూడా
   ఉంది               -> 0.143041
   వుంది              -> 0.114688
   వచ్చింది           -> 0.104315
   మన                 -> 0.102788
Selected sample: ఉంది

Completed sentence: ఆయన చాలా మంచి పేరు కూడా ఉంది

['ఆయన', 'చాలా', 'మంచి', 'పేరు', 'కూడా', 'ఉంది']

In [13]:
prefix = "పుస్తకం"
complete_sentence_gru_fluent(prefix, ngram, model, word2idx, idx2word, CONFIG["device"],sample_T= 0.6, max_steps=7)

# He handed her the book and said, "I'm involved in this."

Prefix tokens: ['పుస్తకం']

Step 1 — context: పుస్తకం
   చాలా               -> 0.085637
   నీ                 -> 0.085275
   లో                 -> 0.083611
   తన                 -> 0.083225
Selected sample: తన

Step 2 — context: పుస్తకం తన
   ముఖంమీద            -> 0.788081
   కూడా               -> 0.021193
   ఆమె                -> 0.021193
   అన్నాడు            -> 0.021193
Selected sample: ముఖంమీద

Step 3 — context: తన ముఖంమీద
   విసిరి             -> 0.753537
   కూడా               -> 0.022406
   ఆమె                -> 0.022406
   అన్నాడు            -> 0.022406
Selected sample: విసిరి

Step 4 — context: ముఖంమీద విసిరి
   కొట్టినప్పుడు      -> 0.495655
   కూడా               -> 0.045850
   ఆమె                -> 0.045850
   అన్నాడు            -> 0.045850
Selected sample: కొట్టినప్పుడు

Step 5 — context: విసిరి కొట్టినప్పుడు
   తనెంత              -> 0.670613
   కూడా               -> 0.029945
   ఆమె                -> 0.029945
   అన్నాడు            -> 0.029945
Selected sample: తనెంత

Step 6 —

['పుస్తకం',
 'తన',
 'ముఖంమీద',
 'విసిరి',
 'కొట్టినప్పుడు',
 'తనెంత',
 'హర్టయింది',
 'తరువాత']